In [1]:
# 데이터 불러오기
import pandas as pd

file_path = "/content/drive/MyDrive/JeonseGuard/실거래가/매매/아파트/202501_아파트_매매_실거래가.csv" # CSV 파일 경로 지정
df = pd.read_csv(file_path, encoding='cp949') # CP949 인코딩

In [2]:
# 상위 5개 확인
df.head()

,NO,시군구,번지,본번,부번,단지명,전용면적(㎡),계약년월,계약일,거래금액(만원),...,층,매수자,매도자,건축년도,도로명,해제사유발생일,거래유형,중개사소재지,등기일자,주택유형
0,1,부산광역시 사하구 괴정동,314-42,314,42,목련(B),48.2000,202501,31,"13,000",...,7,개인,개인,2002,낙동대로140번길 27,-,중개거래,부산 강서구,25.04.08,아파트
1,2,부산광역시 해운대구 재송동,1355,1355,0,센텀리슈빌2단지,59.8934,202501,31,"47,300",...,8,개인,개인,2019,해운대로76번길 40,20250203,중개거래,부산 해운대구,-,아파트
2,3,대구광역시 중구 대신동,2110,2110,0,e편한세상대신,84.9760,202501,31,"51,500",...,15,개인,개인,2018,달구벌대로 1943,-,중개거래,대구 중구,25.03.26,아파트
3,4,서울특별시 강남구 자곡동,602,602,0,강남자곡아이파크,59.9600,202501,31,"134,000",...,7,개인,개인,2014,자곡로 175,-,중개거래,서울 강남구,25.03.28,아파트
4,5,광주광역시 광산구 산월동,882-1,882,1,부영1차,84.9763,202501,31,"25,900",...,7,개인,개인,2003,월계로 170,-,중개거래,광주 광산구,25.02.17,아파트


In [3]:
# 컬럼명 확인
print(df.columns)

Index(['NO', '시군구', '번지', '본번', '부번', '단지명', '전용면적(㎡)', '계약년월', '계약일',
       '거래금액(만원)', '동', '층', '매수자', '매도자', '건축년도', '도로명', '해제사유발생일', '거래유형',
       '중개사소재지', '등기일자', '주택유형'],
      dtype='object')


In [4]:
# 특정 컬럼 값만 확인
df["거래금액(만원)"].head()

,거래금액(만원)
0,"13,000"
1,"47,300"
2,"51,500"
3,"134,000"
4,"25,900"


In [5]:
# 변경 매핑 딕셔너리 정의
renamed_columns = {
    "시군구": "address",
    "본번": "bun",
    "부번": "ji",
    "층": "floor",
    "전용면적(㎡)": "area",
    "계약년월": "contract_year_month",
    "거래금액(만원)": "price",
    "주택유형": "housing_type"
}

In [6]:
# 매핑에 해당하는 컬럼만 필터링
df = df.rename(columns=renamed_columns)
sale_df = df[list(renamed_columns.values())].copy()
sale_df.head()

,address,bun,ji,floor,area,contract_year_month,price,housing_type
0,부산광역시 사하구 괴정동,314,42,7,48.2000,202501,"13,000",아파트
1,부산광역시 해운대구 재송동,1355,0,8,59.8934,202501,"47,300",아파트
2,대구광역시 중구 대신동,2110,0,15,84.9760,202501,"51,500",아파트
3,서울특별시 강남구 자곡동,602,0,7,59.9600,202501,"134,000",아파트
4,광주광역시 광산구 산월동,882,1,7,84.9763,202501,"25,900",아파트


In [7]:
# 쉼표 제거 및 문자열을 정수로 변환
sale_df["price"] = (
    sale_df["price"]
    .astype(str) # 문자열로 변환 (안전)
    .str.replace(",", "") # 쉼표 제거
    .astype(int) # 정수형으로 변환
    * 10000 # 만원 → 원 변환
)

In [8]:
# 변환된 price 컬럼 확인
sale_df[["price"]].head()

,price
0,130000000
1,473000000
2,515000000
3,1340000000
4,259000000


In [9]:
# price 컬럼의 값을 쉼표가 포함된 문자열로 변환
sale_df["price"] = sale_df["price"].apply(lambda x: format(x, ","))
sale_df.head()

,address,bun,ji,floor,area,contract_year_month,price,housing_type
0,부산광역시 사하구 괴정동,314,42,7,48.2000,202501,"130,000,000",아파트
1,부산광역시 해운대구 재송동,1355,0,8,59.8934,202501,"473,000,000",아파트
2,대구광역시 중구 대신동,2110,0,15,84.9760,202501,"515,000,000",아파트
3,서울특별시 강남구 자곡동,602,0,7,59.9600,202501,"1,340,000,000",아파트
4,광주광역시 광산구 산월동,882,1,7,84.9763,202501,"259,000,000",아파트


In [10]:
# 결측치 확인
missing_counts = sale_df.isnull().sum()
print("📌 결측치가 있는 컬럼: ", missing_counts[missing_counts > 0])

📌 결측치가 있는 컬럼:  Series([], dtype: int64)


In [11]:
# 전체 중복된 행의 수 확인
duplicate_count = sale_df.duplicated().sum()
print(f"📌 중복된 행의 수: {duplicate_count}")

📌 중복된 행의 수: 1612


In [12]:
# 중복 제거 (기본: 모든 열 기준, keep='first')
sale_df = sale_df.drop_duplicates().reset_index(drop=True)

In [13]:
# INSERT 구문 생성
insert_header = """INSERT INTO sale_transaction_apartment (address, bun, ji, floor, area, contract_year_month, price, housing_type, created_at, updated_at)
VALUES """

In [14]:
# 행별로 SQL 값 문자열 생성
values = []

In [15]:
# 각 행을 values 리스트에 추가
for _, row in sale_df.iterrows():
    values.append(f"('{row['address']}', '{row['bun']}', '{row['ji']}', '{row['floor']}', '{row['area']}', '{row['contract_year_month']}', '{row['price']}', '{row['housing_type']}', NOW(), NOW())")

In [16]:
# INSERT 구문 상위 5개만 출력
preview_sql = insert_header + ",\n       ".join(values[:5]) + ";"
print(preview_sql)

INSERT INTO sale_transaction_apartment (address, bun, ji, floor, area, contract_year_month, price, housing_type, created_at, updated_at)
VALUES ('부산광역시 사하구 괴정동', '314', '42', '7', '48.2', '202501', '130,000,000', '아파트', NOW(), NOW()),
       ('부산광역시 해운대구 재송동', '1355', '0', '8', '59.8934', '202501', '473,000,000', '아파트', NOW(), NOW()),
       ('대구광역시 중구 대신동', '2110', '0', '15', '84.976', '202501', '515,000,000', '아파트', NOW(), NOW()),
       ('서울특별시 강남구 자곡동', '602', '0', '7', '59.96', '202501', '1,340,000,000', '아파트', NOW(), NOW()),
       ('광주광역시 광산구 산월동', '882', '1', '7', '84.9763', '202501', '259,000,000', '아파트', NOW(), NOW());


In [17]:
# INSERT 구문 조립
insert_sql = insert_header + ",\n       ".join(values) + ";"

In [18]:
# 파일 저장
file_name = "V32__insert_sale_transaction_apartment_202501.sql"

with open(file_name, "w", encoding="utf-8") as f:
    f.write(insert_sql)

print(f"{file_name} 파일이 생성되었습니다.")

V32__insert_sale_transaction_apartment_202501.sql 파일이 생성되었습니다.
